In [ ]:
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.api as sm
from statsmodels.formula.api import ols


df = pd.read_excel("medicine_store_data_Final.xlsx", sheet_name='Purchases')

df = df[['total_price', 'quantity', 'locality']].dropna()

df.rename(columns={'quantity': 'Quantity', 'total_price': 'TotalPrice'}, inplace=True)

bins = [0, 50, 100, 150, 200, 300, 500, 1000]
labels = ['0–50', '51–100', '101–150', '151–200', '201–300', '301–500', '501–1000']
df['PriceBucket'] = pd.cut(df['TotalPrice'], bins=bins, labels=labels)

# ANOVA
model = ols('Quantity ~ C(PriceBucket)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("\n ANOVA Table:")
print(anova_table)

# Post-hoc Test (Tukey's HSD)
df = df.dropna(subset=['PriceBucket'])
tukey = pairwise_tukeyhsd(endog=df['Quantity'], groups=df['PriceBucket'], alpha=0.05)
print("\n Tukey HSD Post-Hoc Results:")
print(tukey.summary())

# Top 2 groups with lowest p-value
tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])
tukey_df = tukey_df.sort_values(by='p-adj')


group1, group2 = tukey_df.iloc[0]['group1'], tukey_df.iloc[0]['group2']
print(f"\n Most distinct pair from Tukey HSD: {group1} vs {group2}")

# t-test on 2 selected bucket
g1_data = df[df['PriceBucket'] == group1]['Quantity']
g2_data = df[df['PriceBucket'] == group2]['Quantity']

t_stat, p_val = ttest_ind(g1_data, g2_data, equal_var=False)

print(f"\n T-test between {group1} and {group2}:")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

print("\n")
print('ANOVA & Post-Hoc Insights: The ANOVA (p≈0.0) confirms significant differences in Quantity across PriceBucket groups.')
print('Tukey\'s HSD highlights "0–50" vs "101–150" as the most distinct pair (p=0.0), supported by an extreme t-statistic (-14.8).')
print('This suggests lower-priced purchases (0–50) drive meaningfully different quantities than mid-range (101–150), likely reflecting')
print('bulk vs single-unit buying behavior.')


In [ ]:
### 2-Way ANOVA

import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Price buckets
def price_bucket(price):
    if price <= 50:
        return '0–50'
    elif price <= 100:
        return '51–100'
    elif price <= 150:
        return '101–150'
    elif price <= 200:
        return '151–200'
    elif price <= 300:
        return '201–300'
    elif price <= 500:
        return '301–500'
    else:
        return '501–1000'

df['PriceBucket'] = df['TotalPrice'].apply(price_bucket)

df_anova = df[['Quantity', 'PriceBucket', 'locality']].dropna()

# Two-way ANOVA
model = ols('Quantity ~ C(PriceBucket) + C(locality) + C(PriceBucket):C(locality)', data=df_anova).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("\n Two-Way ANOVA Table:")
print(anova_table)

print("\n")
print("PriceBucket Dominates: The extremely low p-value (≈0.0) for PriceBucket confirms it significantly affects purchase")
print(" quantity, while locality (p=0.43) and interaction effects (p=0.62) are not statistically significant.\n")
print("No Localized Pricing Patterns: The high p-value for locality and its interaction with PriceBucket suggests no meaningful")
print(" regional differences in how price buckets influence quantity purchased.\n")
print("Actionable Takeaway: Focus pricing strategies on global trends (e.g., discounts for mid-range buckets like 101–150)")
print(" rather than location-specific adjustments.")


In [ ]:
df

In [ ]:
### Delivery Expansion

In [ ]:
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import pandas as pd

top_localities = df['locality'].value_counts().index[:5]
df_bayes = df[df['locality'].isin(top_localities)].copy()

df_bayes['locality_code'] = df_bayes['locality'].astype('category').cat.codes
n_localities = df_bayes['locality_code'].nunique()

locality_map = dict(enumerate(df_bayes['locality'].astype('category').cat.categories))

with pm.Model() as model:
    mu = pm.Normal('mu', mu=0, sigma=10, shape=n_localities)
    sigma = pm.Exponential('sigma', 1.0)

    y_obs = pm.Normal('y_obs', mu=mu[df_bayes['locality_code'].values],
                      sigma=sigma, observed=df_bayes['Quantity'])

    trace = pm.sample(2000, tune=1000, return_inferencedata=True, random_seed=42)

# Plotting
fig, ax = plt.subplots(figsize=(8, 5))
az.plot_forest(trace, var_names=['mu'], combined=True, hdi_prob=0.95, ax=ax,)
locality_names = [locality_map[i] for i in range(n_localities)]
ax.set_yticklabels(locality_names)

plt.title('Bayesian Estimation: Mean Quantity by Locality')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Extracting posterior means for mu
mu_means = trace.posterior['mu'].mean(dim=["chain", "draw"]).values
locality_labels = df_bayes['locality'].astype('category').cat.categories

locality_summary = pd.DataFrame({
    'Locality': locality_labels,
    'PosteriorMean': mu_means
}).sort_values(by='PosteriorMean', ascending=False)

print(locality_summary)


In [ ]:
# Compute HDIs
hdi_95 = az.hdi(trace, hdi_prob=0.95)['mu'].values

locality_summary['Lower_95_HDI'] = hdi_95[:, 0]
locality_summary['Upper_95_HDI'] = hdi_95[:, 1]
locality_summary['HDI_Width'] = locality_summary['Upper_95_HDI'] - locality_summary['Lower_95_HDI']

# Filter: high mean, low uncertainty
top_expansion_targets = locality_summary.sort_values(
    by=['PosteriorMean', 'HDI_Width'], ascending=[False, True]
)

print("Suggested Localities for Delivery Expansion:")
print(top_expansion_targets.head(5))


In [ ]:
### Testing for Khardah's Profitability

In [ ]:
test_df = df[df['locality'].isin(['Sodepur', 'Khardah'])].copy()

test_df['Group'] = test_df['locality'].map({'Sodepur': 'A', 'Khardah': 'B'})

print("Baseline Metrics (Sodepur vs. Khardah):")
print(test_df.groupby('locality').agg({
    'Quantity': ['mean', 'std', 'count'],
    'TotalPrice': ['mean', 'std']
}))


In [ ]:
### T-test and Chi-square to see if Khardah is really not a profitable option

In [ ]:
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind

kh_vs_sp = df[df['locality'].isin(['Khardah','Sodepur'])].copy()

# T-test for Quantity
t_stat, p_val = ttest_ind(kh_vs_sp[kh_vs_sp['locality']=='Khardah']['Quantity'],
                         kh_vs_sp[kh_vs_sp['locality']=='Sodepur']['Quantity'],
                         equal_var=False)
print(f"Khardah vs Sodepur T-test p-value: {p_val:.4f}")

# Chi-square for PriceBuckets
contingency = pd.crosstab(kh_vs_sp['locality'], kh_vs_sp['PriceBucket'])
chi2, p_chi, _, _ = chi2_contingency(contingency)
print(f"PriceBucket Chi-square p-value: {p_chi:.4f}")


In [ ]:
import numpy as np
from scipy.stats import beta

def bayesian_ab_test(group_a, group_b):
    """
    Bayesian A/B test for continuous quantity data (simplified approach).
    Converts continuous values to binary "successes" (above median).
    """
    combined = np.concatenate([group_a, group_b])
    median_val = np.median(combined)

    success_a = np.sum(group_a > median_val)
    success_b = np.sum(group_b > median_val)
    total_a, total_b = len(group_a), len(group_b)

    samples_a = beta.rvs(1 + success_a, 1 + total_a - success_a, size=10000)
    samples_b = beta.rvs(1 + success_b, 1 + total_b - success_b, size=10000)

    return {
        'P(B > A)': np.mean(samples_b > samples_a),
        'Median A': np.median(samples_a),
        'Median B': np.median(samples_b),
        'Uplift': np.median(samples_b - samples_a)
    }

# Prepare data
group_a = df[df['locality'] == 'Sodepur']['Quantity'].values
group_b = df[df['locality'] == 'Khardah']['Quantity'].values

# Run test
results = bayesian_ab_test(group_a, group_b)
print(f"Probability Khardah > Sodepur: {results['P(B > A)']:.1%}")
print(f"Median conversion rate - Sodepur: {results['Median A']:.3f}")
print(f"Median conversion rate - Khardah: {results['Median B']:.3f}")
print(f"Estimated uplift: {results['Uplift']:.3f}")

print("\nStatistically Identical Demand:")
print("Bayesian A/B test shows ~50% probability Khardah outperforms Sodepur (≈50% means no difference).")
print("Near-identical median conversion rates and zero uplift confirm demand is indistinguishable.")
print("\nSupporting Evidence:")
print("T-test (p=0.94) and Chi-square (p=0.99) validate no meaningful differences in quantity or price sensitivity.")


In [ ]:
def bayesian_ab_test_adjacent(group_a, group_b, adjacency_bonus=0.2):
    results = bayesian_ab_test(group_a, group_b)
    adjusted_uplift = results['Uplift'] * (1 + adjacency_bonus)

    return {
        'P(B > A)': results['P(B > A)'],
        'Original Uplift': results['Uplift'],
        'Adjacency Bonus (%)': adjacency_bonus * 100,
        'Adjusted Uplift': adjusted_uplift
    }

# Test Khardah (adjacent) vs Sodepur
results_kh = bayesian_ab_test_adjacent(
    df[df['locality'] == 'Sodepur']['Quantity'].values,
    df[df['locality'] == 'Khardah']['Quantity'].values,
    adjacency_bonus=0.2
)

# Test Agarpara (adjacent) vs Sodepur
results_ag = bayesian_ab_test_adjacent(
    df[df['locality'] == 'Sodepur']['Quantity'].values,
    df[df['locality'] == 'Agarpara']['Quantity'].values,
    adjacency_bonus=0.2
)

print("Khardah vs Sodepur (Adjacent):", results_kh)
print("Agarpara vs Sodepur (Adjacent):", results_ag)

print("\nKhardah vs. Sodepur:")
print("Identical Demand: Low probability of Khardah outperforming Sodepur and negative adjusted uplift confirm no advantage.")
print("\nAgarpara vs. Sodepur:")
print("Clear Winner: High probability Agarpara outperforms Sodepur, with positive adjusted uplift after adjacency bonus.")


In [ ]:
### Delivery Options Comparison

import numpy as np
from scipy.stats import beta
from tabulate import tabulate

def bayesian_ab_test_core(group_a, group_b):
    combined = np.concatenate([group_a, group_b])
    median_val = np.median(combined)
    success_a = np.sum(group_a > median_val)
    success_b = np.sum(group_b > median_val)
    total_a, total_b = len(group_a), len(group_b)

    samples_a = beta.rvs(1 + success_a, 1 + total_a - success_a, size=10000)
    samples_b = beta.rvs(1 + success_b, 1 + total_b - success_b, size=10000)

    return {
        'P(B > A)': np.mean(samples_b > samples_a),
        'Uplift': np.median(samples_b - samples_a)
    }

def logistic_adjusted_uplift(group_a, group_b, option='train_bicycle'):
    base = bayesian_ab_test_core(group_a, group_b)

    params = {
        'train_bicycle': {'cost': 14, 'time_hrs': 3, 'label': 'Train + Bicycle'},
        'bike_only': {'cost': 25, 'time_hrs': 50/60, 'label': 'Bike Only'}
    }[option]

    demand_decay = 1 - (0.02 * params['time_hrs'])
    cost_penalty = 1 - (0.01 * max(0, params['cost'] - 10) / 10)

    adjusted_uplift = base['Uplift'] * demand_decay * cost_penalty
    roi = (adjusted_uplift * 10000) / params['cost']

    return {
        **base,
        'Option': params['label'],
        'Adjusted Uplift': adjusted_uplift,
        'ROI per ₹': roi,
        'Demand Decay (%)': (1 - demand_decay) * 100,
        'Cost Penalty (%)': (1 - cost_penalty) * 100
    }

group_sodepur = df[df['locality'] == 'Sodepur']['Quantity'].values
group_agarpara = df[df['locality'] == 'Khardah']['Quantity'].values

results_train = logistic_adjusted_uplift(group_sodepur, group_agarpara, 'train_bicycle')
results_bike = logistic_adjusted_uplift(group_sodepur, group_agarpara, 'bike_only')

table_data = [
    [
        results_train['Option'],
        f"{results_train['P(B > A)']:.1%}",
        f"{results_train['Uplift']:.5f}",
        f"{results_train['Adjusted Uplift']:.5f}",
        f"{results_train['ROI per ₹']:.3f}",
        f"{results_train['Demand Decay (%)']:.1f}%",
        f"{results_train['Cost Penalty (%)']:.1f}%"
    ],
    [
        results_bike['Option'],
        f"{results_bike['P(B > A)']:.1%}",
        f"{results_bike['Uplift']:.5f}",
        f"{results_bike['Adjusted Uplift']:.5f}",
        f"{results_bike['ROI per ₹']:.3f}",
        f"{results_bike['Demand Decay (%)']:.1f}%",
        f"{results_bike['Cost Penalty (%)']:.1f}%"
    ]
]

headers = ["Option", "P(B > A)", "Original Uplift", "Adjusted Uplift", "ROI per ₹", "Demand Decay", "Cost Penalty"]
print(tabulate(table_data, headers=headers, tablefmt="grid"))


In [ ]:
import matplotlib.pyplot as plt

results_train = logistic_adjusted_uplift(group_sodepur, group_agarpara, 'train_bicycle')
results_bike = logistic_adjusted_uplift(group_sodepur, group_agarpara, 'bike_only')

labels = [results_train['Option'], results_bike['Option']]
roi_values = [results_train['ROI per ₹'], results_bike['ROI per ₹']]
uplift_values = [results_train['Adjusted Uplift'], results_bike['Adjusted Uplift']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(labels, roi_values, color=['green', 'orange'])
ax1.set_ylabel('ROI per ₹1 Spent')
ax1.set_title('Cost Efficiency')
ax1.grid(axis='y', linestyle='--', alpha=0.7)

ax2.bar(labels, uplift_values, color=['green', 'orange'])
ax2.set_ylabel('Adjusted Uplift')
ax2.set_title('Demand Impact')
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()
